# Hotel Booking Cancellation Prediction - Dataset Analysis

## Problem Overview
This dataset contains hotel booking information with the goal of predicting whether a customer will cancel their hotel booking appointment or not.

**Target Variable:**
- 1: Canceled
- 0: Not Canceled

**Evaluation Metric:** Macro-F1

**Dataset Structure:**
- Training data: train.csv (with labels)
- Test data: test.csv (without labels)
- Sample submission: sample_submission.csv


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


## 1. Data Loading and Initial Exploration


In [ ]:
# Load the datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

print("Dataset Shapes:")
print(f"Training data: {train_df.shape}")
print(f"Test data: {test_df.shape}")
print(f"Sample submission: {sample_submission.shape}")
print("\n" + "="*50 + "\n")

print("Training Data Info:")
print(train_df.info())
print("\n" + "="*50 + "\n")

print("First few rows of training data:")
print(train_df.head())


## 2. Data Quality Assessment


In [ ]:
# Check for missing values
print("Missing Values in Training Data:")
missing_train = train_df.isnull().sum()
print(missing_train[missing_train > 0])

print("\nMissing Values in Test Data:")
missing_test = test_df.isnull().sum()
print(missing_test[missing_test > 0])

print("\n" + "="*50 + "\n")

# Check data types
print("Data Types:")
print(train_df.dtypes)

print("\n" + "="*50 + "\n")

# Check for duplicates
print(f"Duplicate rows in training data: {train_df.duplicated().sum()}")
print(f"Duplicate rows in test data: {test_df.duplicated().sum()}")


## 3. Target Variable Analysis


In [ ]:
# Analyze target variable distribution
print("Target Variable Distribution:")
label_counts = train_df['label'].value_counts()
print(label_counts)
print(f"\nPercentage distribution:")
print(label_counts / len(train_df) * 100)

# Visualize target distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
label_counts.plot(kind='bar', ax=ax1, color=['skyblue', 'lightcoral'])
ax1.set_title('Distribution of Hotel Booking Cancellations')
ax1.set_xlabel('Label (0: Not Canceled, 1: Canceled)')
ax1.set_ylabel('Count')
ax1.set_xticklabels(['Not Canceled', 'Canceled'], rotation=0)

# Pie chart
label_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=['skyblue', 'lightcoral'])
ax2.set_title('Percentage Distribution of Cancellations')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

print(f"\nClass imbalance ratio: {label_counts[1] / label_counts[0]:.3f}")


## 4. Feature Analysis - Categorical Variables


In [ ]:
# Identify categorical variables
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical variables: {categorical_cols}")

# Analyze each categorical variable
for col in categorical_cols:
    print(f"\n{col.upper()}:")
    print(f"Unique values: {train_df[col].nunique()}")
    print(f"Value counts:")
    print(train_df[col].value_counts())
    
    # Calculate cancellation rate by category
    cancellation_rate = train_df.groupby(col)['label'].mean()
    print(f"\nCancellation rate by {col}:")
    print(cancellation_rate.sort_values(ascending=False))
    print("-" * 50)


## 5. Feature Analysis - Numerical Variables


In [ ]:
# Identify numerical variables
numerical_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols = [col for col in numerical_cols if col not in ['id', 'label']]
print(f"Numerical variables: {numerical_cols}")

# Statistical summary
print("\nStatistical Summary of Numerical Variables:")
print(train_df[numerical_cols].describe())

# Check for outliers using IQR method
print("\nOutlier Analysis (using IQR method):")
for col in numerical_cols:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = train_df[(train_df[col] < lower_bound) | (train_df[col] > upper_bound)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(train_df)*100:.2f}%)")


## 6. Correlation Analysis


In [ ]:
# Create correlation matrix for numerical variables
correlation_matrix = train_df[numerical_cols + ['label']].corr()

# Plot correlation heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of Numerical Variables')
plt.tight_layout()
plt.show()

# Show correlations with target variable
print("\nCorrelations with Target Variable (label):")
correlations_with_target = correlation_matrix['label'].drop('label').sort_values(key=abs, ascending=False)
print(correlations_with_target)


## 7. Feature Distribution Analysis


In [ ]:
# Plot distributions of key numerical features
key_features = ['lead_time', 'avg_price_per_room', 'no_of_adults', 'no_of_children', 
                'no_of_weekend_nights', 'no_of_week_nights']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, feature in enumerate(key_features):
    if feature in train_df.columns:
        # Plot distribution by cancellation status
        for label_val in [0, 1]:
            data = train_df[train_df['label'] == label_val][feature]
            axes[i].hist(data, alpha=0.7, label=f'Label {label_val}', bins=30)
        
        axes[i].set_title(f'Distribution of {feature} by Cancellation Status')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Frequency')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Categorical Feature Analysis with Visualizations


In [ ]:
# Plot cancellation rates for categorical variables
categorical_cols_to_plot = ['type_of_meal_plan', 'room_type_reserved', 'market_segment_type']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, col in enumerate(categorical_cols_to_plot):
    if col in train_df.columns:
        # Calculate cancellation rate by category
        cancellation_by_category = train_df.groupby(col)['label'].agg(['count', 'mean']).reset_index()
        cancellation_by_category.columns = [col, 'count', 'cancellation_rate']
        
        # Sort by cancellation rate
        cancellation_by_category = cancellation_by_category.sort_values('cancellation_rate', ascending=False)
        
        # Create bar plot
        bars = axes[i].bar(range(len(cancellation_by_category)), 
                          cancellation_by_category['cancellation_rate'],
                          color='skyblue', alpha=0.7)
        
        axes[i].set_title(f'Cancellation Rate by {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Cancellation Rate')
        axes[i].set_xticks(range(len(cancellation_by_category)))
        axes[i].set_xticklabels(cancellation_by_category[col], rotation=45, ha='right')
        axes[i].grid(True, alpha=0.3)
        
        # Add value labels on bars
        for j, bar in enumerate(bars):
            height = bar.get_height()
            axes[i].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 9. Time-based Analysis


In [ ]:
# Analyze cancellation patterns by time
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Cancellation rate by arrival year
year_cancellation = train_df.groupby('arrival_year')['label'].mean()
axes[0, 0].bar(year_cancellation.index, year_cancellation.values, color='lightblue')
axes[0, 0].set_title('Cancellation Rate by Arrival Year')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Cancellation Rate')
axes[0, 0].grid(True, alpha=0.3)

# Cancellation rate by arrival month
month_cancellation = train_df.groupby('arrival_month')['label'].mean()
axes[0, 1].bar(month_cancellation.index, month_cancellation.values, color='lightgreen')
axes[0, 1].set_title('Cancellation Rate by Arrival Month')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Cancellation Rate')
axes[0, 1].grid(True, alpha=0.3)

# Lead time distribution by cancellation status
for label_val in [0, 1]:
    data = train_df[train_df['label'] == label_val]['lead_time']
    axes[1, 0].hist(data, alpha=0.7, label=f'Label {label_val}', bins=30)
axes[1, 0].set_title('Lead Time Distribution by Cancellation Status')
axes[1, 0].set_xlabel('Lead Time (days)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Average price per room by cancellation status
price_by_label = train_df.groupby('label')['avg_price_per_room'].mean()
axes[1, 1].bar(['Not Canceled', 'Canceled'], price_by_label.values, color=['lightcoral', 'skyblue'])
axes[1, 1].set_title('Average Price per Room by Cancellation Status')
axes[1, 1].set_ylabel('Average Price')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 10. Key Insights and Summary


In [ ]:
# Generate summary statistics
print("=== DATASET ANALYSIS SUMMARY ===\n")

print(f"Dataset Size:")
print(f"- Training samples: {len(train_df):,}")
print(f"- Test samples: {len(test_df):,}")
print(f"- Features: {len(train_df.columns) - 2}")

print(f"\nTarget Variable Distribution:")
print(f"- Not Canceled (0): {label_counts[0]:,} ({label_counts[0]/len(train_df)*100:.1f}%)")
print(f"- Canceled (1): {label_counts[1]:,} ({label_counts[1]/len(train_df)*100:.1f}%)")
print(f"- Class imbalance ratio: {label_counts[1] / label_counts[0]:.3f}")

print(f"\nData Quality:")
print(f"- Missing values in training: {train_df.isnull().sum().sum()}")
print(f"- Missing values in test: {test_df.isnull().sum().sum()}")
print(f"- Duplicate rows in training: {train_df.duplicated().sum()}")
print(f"- Duplicate rows in test: {test_df.duplicated().sum()}")

print(f"\nTop Correlated Features with Target:")
top_correlations = correlations_with_target.head(5)
for feature, corr in top_correlations.items():
    print(f"- {feature}: {corr:.3f}")

print(f"\nKey Insights:")
print(f"- This is a binary classification problem with moderate class imbalance")
print(f"- The dataset contains both numerical and categorical features")
print(f"- No missing values detected in the dataset")
print(f"- Lead time and price appear to be important predictors")
print(f"- Evaluation metric: Macro-F1 score")

# Save the analysis
print(f"\nAnalysis completed. Ready for model development!")
